In [2]:
# ===============================
# LINEAR REGRESSION - HOUSING DATA
# ===============================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load the housing dataset
df = pd.read_csv("../data/linearregression.csv")

# 2. Features and target
X = df.drop("price_k", axis=1)
y = df["price_k"]

# 3. Train/Validation/Test split (70% / 15% / 15%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Shapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

# 4. Identify numeric and categorical columns
numeric_cols = X_train.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# 5. Handle missing values
# Numeric imputation
num_imputer = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[numeric_cols]), columns=numeric_cols, index=X_train.index)
X_val_num   = pd.DataFrame(num_imputer.transform(X_val[numeric_cols]), columns=numeric_cols, index=X_val.index)
X_test_num  = pd.DataFrame(num_imputer.transform(X_test[numeric_cols]), columns=numeric_cols, index=X_test.index)

# Categorical imputation
cat_imputer = SimpleImputer(strategy="most_frequent")
X_train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train[categorical_cols]), columns=categorical_cols, index=X_train.index)
X_val_cat   = pd.DataFrame(cat_imputer.transform(X_val[categorical_cols]), columns=categorical_cols, index=X_val.index)
X_test_cat  = pd.DataFrame(cat_imputer.transform(X_test[categorical_cols]), columns=categorical_cols, index=X_test.index)

# 6. Encoding categorical features
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat_enc = pd.DataFrame(encoder.fit_transform(X_train_cat), columns=encoder.get_feature_names_out(categorical_cols), index=X_train.index)
X_val_cat_enc   = pd.DataFrame(encoder.transform(X_val_cat), columns=encoder.get_feature_names_out(categorical_cols), index=X_val.index)
X_test_cat_enc  = pd.DataFrame(encoder.transform(X_test_cat), columns=encoder.get_feature_names_out(categorical_cols), index=X_test.index)

# 7. Combine numeric + encoded categorical
X_train_final = pd.concat([X_train_num, X_train_cat_enc], axis=1)
X_val_final   = pd.concat([X_val_num, X_val_cat_enc], axis=1)
X_test_final  = pd.concat([X_test_num, X_test_cat_enc], axis=1)

print("Final shapes after encoding:")
print("X_train_final:", X_train_final.shape)
print("X_val_final  :", X_val_final.shape)
print("X_test_final :", X_test_final.shape)

# 8. Linear Regression model
lr = LinearRegression()

# 9. RandomizedSearchCV for minor hyperparameter tuning (fit_intercept)
param_dist = {"fit_intercept": [True, False], "positive": [True, False]}
kfold = KFold(n_splits=8, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=lr,
    param_distributions=param_dist,
    n_iter=4,  # small number since only a couple hyperparameters
    cv=kfold,
    scoring='r2',
    n_jobs=1,
    verbose=1,
    random_state=42,
    refit=True
)

random_search.fit(X_train_final, y_train)

best_lr = random_search.best_estimator_
print("\nBest hyperparameters:", random_search.best_params_)
print("Best CV R²:", random_search.best_score_)

# 10. Predictions
y_train_pred = best_lr.predict(X_train_final)
y_val_pred   = best_lr.predict(X_val_final)
y_test_pred  = best_lr.predict(X_test_final)

# 11. Evaluation
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"\n{name} Results")
    print(f"MAE: {mae:.3f}, RMSE: {rmse:.3f}, R²: {r2:.4f}")

evaluate(y_train, y_train_pred, "TRAIN")


Shapes:
X_train: (7000, 7)
X_val  : (1500, 7)
X_test : (1500, 7)
Numeric columns: ['square_feet', 'num_bedrooms', 'num_bathrooms', 'lot_size', 'year_built']
Categorical columns: ['has_garage', 'neighborhood']
Final shapes after encoding:
X_train_final: (7000, 11)
X_val_final  : (1500, 11)
X_test_final : (1500, 11)
Fitting 8 folds for each of 4 candidates, totalling 32 fits

Best hyperparameters: {'positive': True, 'fit_intercept': True}
Best CV R²: 0.9525041869996762

TRAIN Results
MAE: 7.917, RMSE: 9.901, R²: 0.9528


In [3]:
evaluate(y_val, y_val_pred, "VALIDATION")


VALIDATION Results
MAE: 7.768, RMSE: 9.896, R²: 0.9524


In [4]:
evaluate(y_test, y_test_pred, "TEST")

# 12. Coefficients
coeffs = pd.DataFrame({
    "feature": X_train_final.columns,
    "coefficient": best_lr.coef_
}).sort_values(by="coefficient", ascending=False)

print("\nCoefficients:")
print(coeffs)


TEST Results
MAE: 8.114, RMSE: 10.081, R²: 0.9515

Coefficients:
           feature  coefficient
7   neighborhood_A    30.457363
8   neighborhood_B    20.240627
6   has_garage_Yes    14.841965
1     num_bedrooms     9.985730
9   neighborhood_C     9.930980
2    num_bathrooms     7.789457
4       year_built     0.297203
0      square_feet     0.080122
3         lot_size     0.005082
5    has_garage_No     0.000000
10  neighborhood_D     0.000000
